# ភាគបន្ថែម A (ផ្នែក ១)៖ ការណែនាំអំពី PyTorch

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/appendix-A/01_main-chapter-code/code-part1.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ១. ការពិនិត្យ GPU
- ## ២. ការបង្កើត tensor និងចែករំលែកអង្គចងចាំ
- ## ៣. dtype និងការបំលែង
- ## ៤. shape, reshape, view និង matmul
- ## ៥. Activation និង autograd
- ## ៦. ក្លាស NeuralNetwork
- ## ៧. Weight initialization និង no_grad
- ## ៨. ទិន្នន័យឧទាហរណ់
- ## ៩. ក្លាស Dataset
- ## ១០. DataLoader និង batching
- ## ១១. ការវាយតម្លៃម៉ូដែល
- ## ១២. អនុគមន៍ compute_accuracy
- ## ១៣. ការរក្សាទុក និងផ្ទុកម៉ូដែល

## ១. ការពិនិត្យ GPU

សាកល្បងថា torch មើលឃើញ GPU ឬទេ។

In [ ]:
import torch

print(torch.__version__)

In [ ]:
print(torch.cuda.is_available())

## ២. ការបង្កើត tensor និងចែករំលែកអង្គចងចាំ

បង្ហាញថា view/share memory ធ្វើឱ្យការកែមួយប៉ះមួយទៀត។

In [ ]:
import torch
import numpy as np

# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)

# create a 1D tensor (vector) from a Python list
tensor1d = torch.tensor([1, 2, 3])

# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1, 2], 
                         [3, 4]])

# create a 3D tensor from a nested Python list
tensor3d_1 = torch.tensor([[[1, 2], [3, 4]], 
                           [[5, 6], [7, 8]]])

# create a 3D tensor from NumPy array
ary3d = np.array([[[1, 2], [3, 4]], 
                  [[5, 6], [7, 8]]])
tensor3d_2 = torch.tensor(ary3d)  # Copies NumPy array
tensor3d_3 = torch.from_numpy(ary3d)  # Shares memory with NumPy array

In [ ]:
ary3d[0, 0, 0] = 999
print(tensor3d_2) # remains unchanged

In [ ]:
print(tensor3d_3) # changes because of memory sharing

## ៣. dtype និងការបំលែង

បំលែងរវាង int និង float tensors។

In [ ]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

In [ ]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

## ៤. shape, reshape, view និង matmul

ប្រតិបត្តិការរូបរាង និងការគុណម៉ាទ្រីស។

In [ ]:
tensor2d = torch.tensor([[1, 2, 3], 
                         [4, 5, 6]])
tensor2d

In [ ]:
tensor2d.shape

In [ ]:
tensor2d.reshape(3, 2)

In [ ]:
tensor2d.view(3, 2)

In [ ]:
tensor2d.T

In [ ]:
tensor2d.matmul(tensor2d.T)

In [ ]:
tensor2d @ tensor2d.T

## ៥. Activation និង autograd

គណនា loss ហើយប្រើ backward() ដើម្បីទាញយក gradients។

In [ ]:
import torch.nn.functional as F

y = torch.tensor([1.0])  # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0])  # bias unit

z = x1 * w1 + b          # net input
a = torch.sigmoid(z)     # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b 
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

In [ ]:
loss.backward()

print(w1.grad)
print(b.grad)

## ៦. ក្លាស NeuralNetwork

សាងសង់ nn.Module ផ្ទាល់ខ្លួនដំបូង។

In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
                
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [ ]:
model = NeuralNetwork(50, 3)

In [ ]:
print(model)

In [ ]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

In [ ]:
print(model.layers[0].weight)

## ៧. Weight initialization និង no_grad

ពិនិត្យ weights និងបិទ gradient ពេលមិនត្រូវការ។

In [ ]:
torch.manual_seed(123)

model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

In [ ]:
print(model.layers[0].weight.shape)

In [ ]:
torch.manual_seed(123)

X = torch.rand((1, 50))
out = model(X)
print(out)

In [ ]:
with torch.no_grad():
    out = model(X)
print(out)

In [ ]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

## ៨. ទិន្នន័យឧទាហរណ់

បង្កើត X_train/X_test និង labels។

In [ ]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [ ]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

## ៩. ក្លាស Dataset

សាងសង់ ToyDataset តាម interface របស់ PyTorch។

In [ ]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]        
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
len(train_ds)

## ១០. DataLoader និង batching

បង្កើត batches ដោយ shuffle និង batch_size។

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

In [ ]:
test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [ ]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

In [ ]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [ ]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

## ១១. ការវាយតម្លៃម៉ូដែល

ដំណើរការ forward ក្នុង eval mode និងប្រៀបធៀងការព្យាករណ៍។

In [ ]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)
        
        loss = F.cross_entropy(logits, labels) # Loss function
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

In [ ]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

In [ ]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

In [ ]:
predictions == y_train

In [ ]:
torch.sum(predictions == y_train)

## ១២. អនុគមន៍ compute_accuracy

គណនា accuracy លើ dataloader ណាមួយ។

In [ ]:
def compute_accuracy(model, dataloader):

    model.eval()
    correct = 0.0
    total_examples = 0
    
    for idx, (features, labels) in enumerate(dataloader):
        
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [ ]:
compute_accuracy(model, train_loader)

In [ ]:
compute_accuracy(model, test_loader)

## ១៣. ការរក្សាទុក និងផ្ទុកម៉ូដែល

រក្សាទុក state_dict ហើយផ្ទុកវិញចូលម៉ូដែលថ្មី។

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ